# Validierung der politischen Achse

In [1]:
source("setup.R")

data.table 1.17.8 using 8 threads (see ?getDTthreads).  Latest news: r-datatable.com

Attaching package: ‘igraph’

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union


Attaching package: ‘dbscan’

The following object is masked from ‘package:stats’:

    as.dendrogram

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::%--%()      masks igraph::%--%()
✖ dplyr::as_data_frame() masks tibble::as_data_frame(), igraph::as_data_frame()
✖ dplyr::between()       masks data.table::between()
✖ purrr::compose()       masks igraph::compose()
✖ tidyr::crossing()      masks igraph::crossing()
✖ dplyr::filte

Warning messages:
1: package ‘igraph’ was built under R version 4.5.3 
2: package ‘ineq’ was built under R version 4.5.2 


## Achse, Scores, Partition

In [2]:
raum_normiert <- function(pfad) {
  v <- fread(pfad, skip = 1, header = FALSE)
  setnames(v, 1, "sub")
  M <- as.matrix(v[, -1])
  M <- M[, colSums(is.na(M)) < nrow(M), drop = FALSE]
  rownames(M) <- v$sub
  M / sqrt(rowSums(M^2))
}

# Achse aus Seed-Paaren: Summe der Differenzvektoren rechts minus links. Für die
# Richtung ist es gleich, ob summiert oder gemittelt wird, normiert wird ohnehin.
achse_bauen <- function(seeds, M) {
  da <- Filter(function(p) all(p %in% rownames(M)), seeds)
  R  <- M[vapply(da, function(p) p[2], ""), , drop = FALSE]
  L  <- M[vapply(da, function(p) p[1], ""), , drop = FALSE]
  list(vektor = colSums(R - L), n_paare = length(da),
       fehlend = Filter(function(p) !all(p %in% rownames(M)), seeds))
}

normieren <- function(v) v / sqrt(sum(v^2))

SEEDS <- list(
  c("Liberal", "Conservative"), c("progressive", "conservatives"),
  c("Democrat", "Republican"), c("Political_Revolution", "ConservativesOnly"),
  c("AskALiberal", "askaconservative"), c("AskDemocrats", "AskTrumpSupporters"),
  c("askhillarysupporters", "AskThe_Donald"), c("hillaryclinton", "The_Donald"),
  c("SandersForPresident", "HillaryForPrison"), c("Impeach_Trump", "HillaryMeltdown"))

M16 <- raum_normiert(vek_datei(2016))
cat("Vektoren 2016:", nrow(M16), "x", ncol(M16), "\n")

achse_eigen <- achse_bauen(SEEDS, M16)
score_eigen <- as.vector(M16 %*% normieren(achse_eigen$vektor))
names(score_eigen) <- rownames(M16)
cat(sprintf("Eigene Achse: %d von %d Seed-Paaren\n", achse_eigen$n_paare, length(SEEDS)))

cl16 <- read_csv(CLUSTER_CSV, show_col_types = FALSE) |>
  filter(jahr == 2016, cluster != -1) |>
  select(subreddit, cluster)
cat("Fixe Partition 2016:", n_distinct(cl16$cluster), "Cluster,", nrow(cl16), "Subreddits\n")

cat("\nAnker auf der eigenen Achse, positiv ist rechts:\n")
print(round(score_eigen[c("Conservative", "The_Donald", "politics",
                          "SandersForPresident", "hillaryclinton")], 3))

Vektoren 2016: 16618 x 150 
Eigene Achse: 10 von 10 Seed-Paaren
Fixe Partition 2016: 95 Cluster, 10941 Subreddits

Anker auf der eigenen Achse, positiv ist rechts:
       Conservative          The_Donald            politics SandersForPresident 
              0.419               0.363              -0.047              -0.266 
     hillaryclinton 
             -0.346 


## Robustheit gegen die Seed-Wahl

In [3]:
WA_PARTISAN <- list(
  c("democrats", "Conservative"), c("GunsAreCool", "progun"),
  c("OpenChristian", "TrueChristian"), c("GamerGhazi", "KotakuInAction"),
  c("excatholic", "Catholicism"), c("EnoughLibertarianSpam", "ShitRConservativeSays"),
  c("AskAnAmerican", "askaconservative"), c("askhillarysupporters", "AskTrumpSupporters"),
  c("liberalgunowners", "Firearms"), c("lastweektonight", "CGPGrey"))

WA_PARTISAN_B <- list(
  c("hillaryclinton", "The_Donald"), c("GamerGhazi", "KotakuInAction"),
  c("SandersForPresident", "HillaryForPrison"), c("askhillarysupporters", "AskThe_Donald"),
  c("BlueMidterm2018", "PoliticalHumor"), c("badwomensanatomy", "ChoosingBeggars"),
  c("PoliticalVideo", "uncensorednews"), c("liberalgunowners", "Firearms"),
  c("GrassrootsSelect", "DNCleaks"), c("GunsAreCool", "dgu"))

kosinus <- function(a, b) sum(a * b) / sqrt(sum(a^2) * sum(b^2))

for (nm in c("partisan", "partisan_b")) {
  seeds <- if (nm == "partisan") WA_PARTISAN else WA_PARTISAN_B
  d <- achse_bauen(seeds, M16)
  cat(sprintf("Waller/Anderson %-11s: %d von %d Paaren | Kosinus zur eigenen Achse = %.3f\n",
              nm, d$n_paare, length(seeds), kosinus(achse_eigen$vektor, d$vektor)))
  if (length(d$fehlend))
    cat("   nicht im Raum:",
        paste(vapply(d$fehlend, paste, "", collapse = "/"), collapse = ", "), "\n")
}

Waller/Anderson partisan   : 10 von 10 Paaren | Kosinus zur eigenen Achse = 0.722
Waller/Anderson partisan_b : 9 von 10 Paaren | Kosinus zur eigenen Achse = 0.683
   nicht im Raum: GrassrootsSelect/DNCleaks 


## Placebo-Achsen: das Niveau

In [4]:
# Die Cluster sind kompakte Gebiete im selben Raum, fast jede Richtung
# trennt sie einigermassen. Das Niveau taugt daher nur als untere Schranke.
eta2_achse <- function(score, codes, k) {
  gm   <- mean(score)
  sst  <- sum((score - gm)^2)
  sums <- as.vector(rowsum(score, codes))
  cnts <- tabulate(codes, nbins = k)
  mittel <- sums / pmax(cnts, 1)
  sum(cnts * (mittel - gm)^2) / sst
}

gemeinsam <- intersect(rownames(M16), cl16$subreddit)
Vc     <- M16[gemeinsam, , drop = FALSE]
codes  <- as.integer(factor(cl16$cluster[match(gemeinsam, cl16$subreddit)]))
k_cl   <- max(codes)
cat("Cluster-Subreddits mit Vektor:", length(gemeinsam), "\n")

w_real <- normieren(achse_eigen$vektor)
real   <- eta2_achse(as.vector(Vc %*% w_real), codes, k_cl)

set.seed(42)
N_ACHSEN <- 1000L
DIM <- ncol(Vc)
Rz  <- matrix(rnorm(N_ACHSEN * DIM), nrow = DIM)
Rz  <- Rz / rep(sqrt(colSums(Rz^2)), each = DIM)
proj_null <- Vc %*% Rz
null <- apply(proj_null, 2, eta2_achse, codes = codes, k = k_cl)

cat(sprintf("Echte Achse eta2 = %.4f (zur Kontrolle: die FF2-Reihe hat 2016 rund 0,16)\n", real))
cat(sprintf("Placebo eta2: Median %.4f | 95 %%-Quantil %.4f | Maximum %.4f | p = %.4f\n",
            median(null), quantile(null, 0.95), max(null), mean(null >= real)))
cat("Liegt die echte Achse unter der Zufallswolke, ist das Niveau nicht politikspezifisch.\n",
    "Das widerlegt die starke Zirkularitaetsthese: Waere die Clusterung um Politik gebaut,\n",
    "muesste ausgerechnet die politische Achse die Cluster am besten trennen. Das Argument\n",
    "liegt also nicht im Niveau, sondern im Anstieg.\n")

Cluster-Subreddits mit Vektor: 10941 
Echte Achse eta2 = 0.1569 (zur Kontrolle: die FF2-Reihe hat 2016 rund 0,16)
Placebo eta2: Median 0.2681 | 95 %-Quantil 0.3605 | Maximum 0.4814 | p = 0.9990
Liegt die echte Achse unter der Zufallswolke, ist das Niveau nicht politikspezifisch.
 Das widerlegt die starke Zirkularitaetsthese: Waere die Clusterung um Politik gebaut,
 muesste ausgerechnet die politische Achse die Cluster am besten trennen. Das Argument
 liegt also nicht im Niveau, sondern im Anstieg.


## Placebo-Achsen: der Anstieg

In [5]:
set.seed(42)
N_TREND <- 500L
# Fünf unabhängige Ziehungen zu je 500 Achsen. Die Kennzahlen einer einzelnen
# Ziehung schwanken merklich, deshalb weist die Zelle unten die Spanne über die
# Blöcke aus. Berichtet wird die Spanne und nicht der Wert einer Ziehung.
N_BLOCK <- 5L
Rt <- matrix(rnorm(N_TREND * N_BLOCK * DIM), nrow = DIM)
Rt <- Rt / rep(sqrt(colSums(Rt^2)), each = DIM)

reihe_echt <- numeric(length(JAHRE))
block      <- rep(seq_len(N_BLOCK), each = N_TREND)
reihe_null <- matrix(NA_real_, nrow = N_TREND * N_BLOCK, ncol = length(JAHRE))

for (yi in seq_along(JAHRE)) {
  j  <- JAHRE[yi]
  Mj <- if (j == 2016) M16 else raum_normiert(ali_datei(j))
  gem <- intersect(rownames(Mj), cl16$subreddit)
  Vj  <- Mj[gem, , drop = FALSE]
  cj  <- as.integer(factor(cl16$cluster[match(gem, cl16$subreddit)]))
  kj  <- max(cj)

  reihe_echt[yi] <- eta2_achse(as.vector(Vj %*% w_real), cj, kj)
  proj <- Vj %*% Rt
  reihe_null[, yi] <- apply(proj, 2, eta2_achse, codes = cj, k = kj)

  cat(sprintf("%d: n = %5d | echt eta2 = %.3f | Zufall Median = %.3f\n",
              j, length(gem), reihe_echt[yi], median(reihe_null[, yi])))
}

delta_echt <- reihe_echt[length(JAHRE)] - reihe_echt[1]
delta_null <- reihe_null[, length(JAHRE)] - reihe_null[, 1]

cat(sprintf("\nAnstieg echte Achse 2016 bis 2024: %+.4f\n", delta_echt))
cat(sprintf("Anstieg Zufallsachsen: Median %+.4f | 95 %%-Quantil %+.4f | Maximum %+.4f\n",
            median(delta_null), quantile(delta_null, 0.95), max(delta_null)))
cat(sprintf("Anteil der Zufallsachsen mit mindestens so grossem Anstieg: %.4f\n",
            mean(delta_null >= delta_echt)))
cat("Die echte Reihe muss die fixe eta2-Reihe aus FF2 treffen, 2016 rund 0,157 und 2024 rund 0,219.\n")

spanne <- function(f, fmt = "%+.4f") {
  w <- vapply(seq_len(N_BLOCK), function(b) f(delta_null[block == b]), numeric(1))
  sprintf(paste0(fmt, " bis ", fmt), min(w), max(w))
}
cat("
Spanne ueber", N_BLOCK, "Ziehungen zu je", N_TREND, "Achsen:
")
cat("  Median          :", spanne(median), "
")
cat("  95 %-Quantil    :", spanne(function(x) quantile(x, 0.95)), "
")
cat("  Anteil >= echt  :", spanne(function(x) mean(x >= delta_echt), "%.3f"), "
")


2016: n = 10941 | echt eta2 = 0.157 | Zufall Median = 0.267
2017: n = 10850 | echt eta2 = 0.185 | Zufall Median = 0.257
2018: n = 10842 | echt eta2 = 0.178 | Zufall Median = 0.259
2019: n = 10820 | echt eta2 = 0.198 | Zufall Median = 0.269
2020: n = 10791 | echt eta2 = 0.192 | Zufall Median = 0.274
2021: n = 10780 | echt eta2 = 0.209 | Zufall Median = 0.277
2022: n = 10754 | echt eta2 = 0.209 | Zufall Median = 0.282
2023: n = 10615 | echt eta2 = 0.188 | Zufall Median = 0.283
2024: n = 10476 | echt eta2 = 0.219 | Zufall Median = 0.280

Anstieg echte Achse 2016 bis 2024: +0.0621
Anstieg Zufallsachsen: Median +0.0115 | 95 %-Quantil +0.0582 | Maximum +0.1033
Anteil der Zufallsachsen mit mindestens so grossem Anstieg: 0.0368
Die echte Reihe muss die fixe eta2-Reihe aus FF2 treffen, 2016 rund 0,157 und 2024 rund 0,219.

Spanne ueber 5 Ziehungen zu je 500 Achsen:
  Median          : +0.0085 bis +0.0136 
  95 %-Quantil    : +0.0570 bis +0.0603 
  Anteil >= echt  : 0.032 bis 0.044 


## Vergleich mit veröffentlichten Werten

In [6]:
WA_URL <- "https://raw.githubusercontent.com/CSSLab/social-dimensions/main/data/scores.csv"

wa <- tryCatch(read_csv(WA_URL, show_col_types = FALSE), error = function(e) NULL)

if (is.null(wa)) {
  cat("Die veroeffentlichten Scores konnten nicht geladen werden (keine Verbindung).\n",
      "Quelle:", WA_URL, "\n")
} else {
  vgl <- wa |>
    select(subreddit = community, wa_partisan = partisan, wa_partisan_b = `partisan B`) |>
    inner_join(tibble(subreddit = names(score_eigen), score_eigen = score_eigen),
               by = "subreddit")
  cat("Überlappende Subreddits:", nrow(vgl), "\n")
  cat("Spearman gegen partisan  :",
      round(cor(vgl$score_eigen, vgl$wa_partisan,   method = "spearman"), 3), "\n")
  cat("Spearman gegen partisan B:",
      round(cor(vgl$score_eigen, vgl$wa_partisan_b, method = "spearman"), 3), "\n")
}

Überlappende Subreddits: 7951 
Spearman gegen partisan  : 0.408 
Spearman gegen partisan B: 0.46 
